# Circle Euler Diagram — Treemap Initialization

Prototype of `treemap_node_positions`: uses a **squarified treemap** to place circles
for a circle-based Euler diagram so that containment is encoded constructively before
gradient descent runs.

**Pipeline**
1. Compute bottom-up leaf-size weights for every node in the inclusion DiGraph.
2. Recursively pack children via the squarified treemap algorithm (Bruls et al. 2000).
3. Each node's initial position = center of its treemap rectangle.
4. Each non-leaf node's initial radius = circumradius of its rectangle (large enough to contain it).

Gradient descent then only needs to tune margins and resolve any partial-overlap
exceptions — it does not need to discover the topology.

In [ ]:
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from collections import Counter

from vizopt.examples.sets import make_british_islands_graph

## Squarified treemap layout

In [ ]:
def _worst_ratio_strip(row, row_sum, W, H, total):
    """Worst aspect ratio among tiles in a candidate strip."""
    if not row or row_sum == 0 or total == 0:
        return float("inf")
    worst = 0.0
    if W >= H:
        strip_w = row_sum / total * W
        for _, w in row:
            tile_h = w / row_sum * H
            if tile_h == 0:
                return float("inf")
            worst = max(worst, max(strip_w, tile_h) / min(strip_w, tile_h))
    else:
        strip_h = row_sum / total * H
        for _, w in row:
            tile_w = w / row_sum * W
            if tile_w == 0:
                return float("inf")
            worst = max(worst, max(tile_w, strip_h) / min(tile_w, strip_h))
    return worst


def _squarify_recursive(items, rect, out):
    """One squarified-treemap step; writes node -> (x0, y0, x1, y1) into *out*."""
    if not items:
        return
    x0, y0, x1, y1 = rect
    W, H = x1 - x0, y1 - y0
    if len(items) == 1:
        out[items[0][0]] = rect
        return
    total = sum(w for _, w in items)
    row, row_sum = [], 0.0
    for name, w in items:
        trial_row = row + [(name, w)]
        trial_sum = row_sum + w
        new_worst = _worst_ratio_strip(trial_row, trial_sum, W, H, total)
        if row:
            curr_worst = _worst_ratio_strip(row, row_sum, W, H, total)
            if curr_worst < new_worst:
                break
        row, row_sum = trial_row, trial_sum
    fraction = row_sum / total
    if W >= H:
        strip_w = fraction * W
        y_cursor = y0
        for name, w in row:
            tile_h = (w / row_sum) * H
            out[name] = (x0, y_cursor, x0 + strip_w, y_cursor + tile_h)
            y_cursor += tile_h
        remaining_rect = (x0 + strip_w, y0, x1, y1)
    else:
        strip_h = fraction * H
        x_cursor = x0
        for name, w in row:
            tile_w = (w / row_sum) * W
            out[name] = (x_cursor, y0, x_cursor + tile_w, y0 + strip_h)
            x_cursor += tile_w
        remaining_rect = (x0, y0 + strip_h, x1, y1)
    _squarify_recursive(items[len(row):], remaining_rect, out)


def squarify_layout(items, rect):
    """Squarified treemap layout (Bruls et al. 2000).

    Args:
        items: sequence of (name, weight) pairs, weights > 0.
        rect: (x0, y0, x1, y1) bounding rectangle.

    Returns:
        Dict mapping name -> (x0, y0, x1, y1).
    """
    if not items:
        return {}
    out = {}
    _squarify_recursive(sorted(items, key=lambda kv: -kv[1]), rect, out)
    return out

## Treemap node positions

In [ ]:
def treemap_node_positions(inclusion_graph, canvas_size=1.0):
    """Treemap-based initial positions for a circle Euler diagram.

    Each node's initial position is the center of its treemap rectangle.
    Each non-leaf node's initial radius is the circumradius of its rectangle
    (the smallest circle containing the rectangle).

    For DAG nodes (two incomparable parents), the first parent in topological
    order wins the node's rectangle assignment.

    Args:
        inclusion_graph: DiGraph with parent->child edges (u->v means v is in u).
            Leaf nodes (out-degree 0) must have a "size" attribute.
        canvas_size: side length of the square canvas.

    Returns:
        Tuple (pos, variable_radii) where:
        - pos: dict node -> (x, y) center
        - variable_radii: dict non-leaf node -> initial radius estimate
    """
    # Bottom-up weights: each node's weight = sum of its leaf descendants' sizes.
    weights = {}
    for node in reversed(list(nx.topological_sort(inclusion_graph))):
        if inclusion_graph.out_degree(node) == 0:
            weights[node] = float(inclusion_graph.nodes[node].get("size", 1.0))
        else:
            child_sum = sum(weights[c] for c in inclusion_graph.successors(node))
            weights[node] = child_sum if child_sum > 0 else 1.0

    roots = [n for n in inclusion_graph.nodes if inclusion_graph.in_degree(n) == 0]
    node_rects = {}

    def _layout_children(parent_rect, children):
        if not children:
            return
        rects = squarify_layout([(c, weights[c]) for c in children], parent_rect)
        for child, child_rect in rects.items():
            if child not in node_rects:  # first-seen wins for DAG nodes
                node_rects[child] = child_rect
            _layout_children(child_rect, list(inclusion_graph.successors(child)))

    canvas_rect = (0.0, 0.0, float(canvas_size), float(canvas_size))
    if len(roots) == 1:
        node_rects[roots[0]] = canvas_rect
        _layout_children(canvas_rect, list(inclusion_graph.successors(roots[0])))
    else:
        root_rects = squarify_layout([(r, weights[r]) for r in roots], canvas_rect)
        for root, rect in root_rects.items():
            node_rects[root] = rect
            _layout_children(rect, list(inclusion_graph.successors(root)))

    pos = {}
    variable_radii = {}
    for node in inclusion_graph.nodes:
        x0, y0, x1, y1 = node_rects.get(node, canvas_rect)
        pos[node] = ((x0 + x1) / 2, (y0 + y1) / 2)
        if inclusion_graph.out_degree(node) > 0:
            variable_radii[node] = float(np.sqrt((x1 - x0) ** 2 + (y1 - y0) ** 2) / 2)

    return pos, variable_radii

## Quick sanity check — British Islands example

The British Islands graph has a known nesting structure 3 levels deep, with one
DAG edge: `Ireland island` contains `Northern Ireland`, which is also inside `United Kingdom`.
Good stress-test for the initialization.

In [ ]:
G = make_british_islands_graph(include_ireland_island=True)

# treemap_node_positions needs "size" on leaf nodes; make_british_islands_graph
# already stores "r" (radius from target_area), so alias it.
for n in G.nodes:
    if G.out_degree(n) == 0:
        G.nodes[n].setdefault("size", G.nodes[n]["r"])

canvas = 10.0
pos, var_radii = treemap_node_positions(G, canvas_size=canvas)

print("Non-leaf nodes and initial radius estimates:")
for name, r in sorted(var_radii.items(), key=lambda kv: -kv[1]):
    print(f"  {name:30s}  r = {r:.3f}")

print()
print("Leaf node positions:")
for n in G.nodes:
    if G.out_degree(n) == 0:
        x, y = pos[n]
        r = G.nodes[n]["r"]
        print(f"  {n:30s}  pos=({x:.2f}, {y:.2f})  r={r:.3f}")

## Visualize the initialization

In [ ]:
def draw_treemap_init(G, pos, var_radii, canvas_size, ax):
    colors = {n: G.nodes[n].get("color", "#999999") for n in G.nodes}
    topo = list(nx.topological_sort(G))

    # Draw non-leaf circles (sets), outermost first so inner ones render on top
    for node in topo:
        if node not in var_radii:
            continue
        cx, cy = pos[node]
        r = var_radii[node]
        ax.add_patch(plt.Circle(
            (cx, cy), r,
            fill=False, edgecolor=colors[node], linewidth=2, linestyle="--", alpha=0.8,
        ))
        ax.text(cx, cy + r * 0.88, node, ha="center", va="top",
                fontsize=7, color=colors[node], weight="bold")

    # Draw leaf circles on top
    for node in G.nodes:
        if G.out_degree(node) != 0:
            continue
        cx, cy = pos[node]
        r = G.nodes[node]["r"]
        ax.add_patch(plt.Circle(
            (cx, cy), r,
            facecolor=colors[node], edgecolor="black", linewidth=0.8, alpha=0.6,
        ))
        ax.text(cx, cy, node, ha="center", va="center", fontsize=6)

    ax.set_xlim(-0.5, canvas_size + 0.5)
    ax.set_ylim(-0.5, canvas_size + 0.5)
    ax.set_aspect("equal")
    ax.set_title("Treemap initialization\n(dashed = non-leaf circle, filled = leaf element)")
    ax.set_xlabel("x")
    ax.set_ylabel("y")


fig, ax = plt.subplots(figsize=(9, 9))
draw_treemap_init(G, pos, var_radii, canvas, ax)
plt.tight_layout()
plt.show()

## Verify containment quality

For each inclusion edge (parent → child), check whether the child circle already
fits inside the parent circle in the treemap initialization:

$$d(\text{parent}, \text{child}) + r_\text{child} \leq r_\text{parent}$$

This should hold for all edges except the DAG edge, where the child got assigned
to its first parent's rectangle.

In [ ]:
def node_radius(node, G, var_radii):
    if node in var_radii:
        return var_radii[node]
    return G.nodes[node].get("r", G.nodes[node].get("size", 1.0))


print(f"{'Edge':52s}  dist+r_child  r_parent  contained  slack")
print("-" * 90)
for parent, child in G.edges:
    px, py = pos[parent]
    cx, cy = pos[child]
    dist = np.sqrt((px - cx) ** 2 + (py - cy) ** 2)
    r_parent = node_radius(parent, G, var_radii)
    r_child  = node_radius(child,  G, var_radii)
    needed = dist + r_child
    slack = r_parent - needed
    ok = slack >= -1e-6
    mark = "✓" if ok else "✗"
    print(f"{parent:25s} -> {child:22s}  {needed:8.3f}      {r_parent:6.3f}  {mark}   {slack:+.3f}")

## Classify all set-pair relations

For each pair of non-leaf nodes classify the relation as:

- **nested** — one set contains the other (graph reachability)
- **disjoint** — no shared leaf descendants  
- **partial overlap** — shared leaves but neither contains the other

This drives the decision of how much work GD will have to do beyond what the treemap already provides.

In [ ]:
def classify_set_pairs(inclusion_graph):
    """Classify every pair of non-leaf nodes as nested / disjoint / partial_overlap.

    Args:
        inclusion_graph: DiGraph with parent->child edges.

    Returns:
        List of (node_a, node_b, relation) triples.
    """
    leaf_set = {n for n in inclusion_graph.nodes if inclusion_graph.out_degree(n) == 0}
    set_nodes = [n for n in inclusion_graph.nodes if n not in leaf_set]

    memberships = {
        n: {d for d in nx.descendants(inclusion_graph, n) if d in leaf_set}
        for n in set_nodes
    }

    results = []
    for i, a in enumerate(set_nodes):
        for b in set_nodes[:i]:
            la, lb = memberships[a], memberships[b]
            if la <= lb or lb <= la:
                relation = "nested"
            elif la.isdisjoint(lb):
                relation = "disjoint"
            else:
                relation = "partial_overlap"
            results.append((a, b, relation))
    return results


pairs = classify_set_pairs(G)
counts = Counter(r for _, _, r in pairs)

print(f"Total set-pairs: {len(pairs)}")
for rel, cnt in sorted(counts.items()):
    print(f"  {rel}: {cnt}")

print()
overlapping = [(a, b) for a, b, r in pairs if r == "partial_overlap"]
if overlapping:
    print("Partial-overlap pairs — need special GD seeding:")
    for a, b in overlapping:
        print(f"  {a}  ∩  {b}")
else:
    print("No partial overlaps — pure nesting, treemap seed is sufficient.")

## Gradient descent

Run the same optimization problem as `optimize_circular_layout_with_enclosed_nodes`,
but seed it from `treemap_node_positions` instead of random positions.

The treemap init already satisfies containment for pure tree edges; GD only needs to:
- grow `British Islands` (the root) slightly to contain `Crown Dependencies` and `Ireland island`
- resolve the `Ireland island ↔ Northern Ireland` DAG violation
- pack glyphs tighter and shrink radii

In [ ]:
from vizopt.base import ObjectiveTerm, OptimizationProblemTemplate, OptimConfig
from vizopt.templates.nested_circles import (
    _compute_collision_pairs,
    _term_total_size,
    _term_collision,
    _term_non_inclusion,
)

inclusion_tree = G
leaf_nodes    = [n for n in inclusion_tree.nodes if inclusion_tree.out_degree(n) == 0]
non_leaf_nodes = [n for n in inclusion_tree.nodes if inclusion_tree.out_degree(n) > 0]
all_node_names = leaf_nodes + non_leaf_nodes

fixed_node_radii = np.array(
    [inclusion_tree.nodes[n].get("size", 1.0) for n in leaf_nodes], dtype=np.float32
)
total_scale = float(fixed_node_radii.sum()) if len(fixed_node_radii) else 10.0

# --- treemap init (our contribution) ---
init_pos, init_var_radii = treemap_node_positions(inclusion_tree, canvas_size=total_scale)
fallback_r = float(fixed_node_radii.max()) if len(fixed_node_radii) else 1.0
initial_node_xys = np.stack(
    [init_pos.get(n, (total_scale / 2, total_scale / 2)) for n in all_node_names],
    dtype=np.float32,
)
initial_variable_radii = np.array(
    [init_var_radii.get(n, fallback_r) for n in non_leaf_nodes], dtype=np.float32
)

# --- problem setup (mirrors optimize_circular_layout_with_enclosed_nodes) ---
node_name_to_id = {name: i for i, name in enumerate(all_node_names)}
inclusion_edge_indices = np.array(
    [(node_name_to_id[u], node_name_to_id[v]) for u, v in inclusion_tree.edges],
    dtype=np.int32,
)
collision_pairs = _compute_collision_pairs(all_node_names, inclusion_tree)

input_parameters = {
    "fixed_node_radii": fixed_node_radii,
    "collision_pairs": collision_pairs,
    "inclusion_edge_indices": inclusion_edge_indices,
}

def initialize(_, _seed):
    return {
        "node_xys": initial_node_xys.copy(),
        "variable_node_radii": initial_variable_radii.copy(),
    }

terms = [
    ObjectiveTerm("total_size",   _term_total_size,   2.0),
    ObjectiveTerm("collision",    _term_collision,     1.0),
    ObjectiveTerm("non_inclusion", _term_non_inclusion, 1.0),
]

problem = OptimizationProblemTemplate(terms=terms, initialize=initialize).instantiate(
    input_parameters
)
optim_vars, history = problem.optimize(OptimConfig(n_iters=4000, learning_rate=0.005))

# --- unpack results ---
pos_gd = {
    node: tuple(float(c) for c in xy)
    for node, xy in zip(all_node_names, optim_vars["node_xys"])
}
var_radii_gd = dict(zip(non_leaf_nodes, [float(r) for r in optim_vars["variable_node_radii"]]))

print("Optimized non-leaf radii:")
for name, r in sorted(var_radii_gd.items(), key=lambda kv: -kv[1]):
    init_r = init_var_radii.get(name, float("nan"))
    print(f"  {name:30s}  init={init_r:.3f}  final={r:.3f}  Δ={r - init_r:+.3f}")

In [ ]:
def draw_circle_layout(G, pos, var_radii, ax, title=""):
    """Draw optimized circle layout (same style as draw_treemap_init)."""
    colors = {n: G.nodes[n].get("color", "#999999") for n in G.nodes}
    # Compute axis limits from positions and radii
    all_cx = [p[0] for p in pos.values()]
    all_cy = [p[1] for p in pos.values()]
    all_r = [var_radii.get(n, G.nodes[n].get("r", 0)) for n in G.nodes]
    margin = max(all_r) * 0.15
    xmin = min(cx - r for cx, r in zip(all_cx, all_r)) - margin
    xmax = max(cx + r for cx, r in zip(all_cx, all_r)) + margin
    ymin = min(cy - r for cy, r in zip(all_cy, all_r)) - margin
    ymax = max(cy + r for cy, r in zip(all_cy, all_r)) + margin

    for node in nx.topological_sort(G):
        if node not in var_radii:
            continue
        cx, cy = pos[node]
        r = var_radii[node]
        ax.add_patch(plt.Circle(
            (cx, cy), r,
            fill=False, edgecolor=colors[node], linewidth=2, linestyle="--", alpha=0.85,
        ))
        ax.text(cx, cy + r * 0.88, node, ha="center", va="top",
                fontsize=7, color=colors[node], weight="bold")

    for node in G.nodes:
        if G.out_degree(node) != 0:
            continue
        cx, cy = pos[node]
        r = G.nodes[node]["r"]
        ax.add_patch(plt.Circle(
            (cx, cy), r,
            facecolor=colors[node], edgecolor="black", linewidth=0.8, alpha=0.65,
        ))
        ax.text(cx, cy, node, ha="center", va="center", fontsize=6)

    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    ax.set_aspect("equal")
    ax.set_title(title)


fig, axes = plt.subplots(1, 2, figsize=(17, 8))

draw_treemap_init(G, pos, var_radii, canvas, axes[0])
axes[0].set_title("Treemap initialization\n(before GD)")

draw_circle_layout(G, pos_gd, var_radii_gd, axes[1],
                   title="After gradient descent\n(3000 iters, lr=0.005)")

plt.tight_layout()
plt.show()

In [ ]:
# Loss history per term
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

term_names = ["total_size", "collision", "non_inclusion"]
for ax, name in zip(axes, term_names):
    iters  = [h["iteration"] for h in history]
    values = [h[name] for h in history]
    ax.plot(iters, values, linewidth=1.5)
    ax.set_title(name)
    ax.set_xlabel("iteration")
    ax.set_ylabel("weighted loss")
    ax.set_yscale("log")
    ax.grid(True, alpha=0.3)

plt.suptitle("Loss history (treemap init → GD)", y=1.02)
plt.tight_layout()
plt.show()

# Final containment check after GD
print("\nFinal containment check:")
print(f"{'Edge':52s}  slack")
print("-" * 65)
for parent, child in G.edges:
    px, py = pos_gd[parent]
    cx, cy = pos_gd[child]
    dist = np.sqrt((px - cx) ** 2 + (py - cy) ** 2)
    r_parent = node_radius(parent, G, var_radii_gd)
    r_child  = node_radius(child,  G, var_radii_gd)
    slack = r_parent - dist - r_child
    mark = "✓" if slack >= -1e-3 else "✗"
    print(f"{parent:25s} -> {child:22s}  {mark}  {slack:+.4f}")

## Greedy bottom-up initialization

Fully greedy, no top-down pass needed:

1. **Leaves** are placed one at a time, largest-first, at the overlap-free position closest to the centroid of all already-placed circles (tries tangent positions at `n_angles` angles around each existing circle).
2. **Non-leaf nodes** are pushed to the *front* of the queue as soon as their last child is placed — before any remaining leaves — so their enclosing circle is added to the placement list and subsequent leaves avoid it.
3. Non-leaf position = centroid of children; radius = enclosing circle + margin.

No top-down pass: positions are absolute from the start.

In [ ]:
from vizopt.templates.nested_circles import greedy_bottomup_node_positions

pos_bu, var_radii_bu = greedy_bottomup_node_positions(G, canvas_size=canvas)

print("Non-leaf nodes and initial radius estimates (greedy bottom-up):")
for name, r in sorted(var_radii_bu.items(), key=lambda kv: -kv[1]):
    r_tm = var_radii.get(name, float("nan"))
    print(f"  {name:30s}  bottom-up={r:.3f}  treemap={r_tm:.3f}  Δ={r - r_tm:+.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 8))

draw_treemap_init(G, pos, var_radii, canvas, axes[0])
axes[0].set_title("Treemap initialization")

draw_treemap_init(G, pos_bu, var_radii_bu, canvas, axes[1])
axes[1].set_title("Greedy bottom-up initialization")

plt.tight_layout()
plt.show()

## Verify containment quality — greedy bottom-up

Same check as before: for each inclusion edge (parent → child), does
`r_parent ≥ dist(parent, child) + r_child`?

In [ ]:
def node_radius_bu(node):
    if node in var_radii_bu:
        return var_radii_bu[node]
    return G.nodes[node].get("r", G.nodes[node].get("size", 1.0))


print(f"{'Edge':52s}  dist+r_child  r_parent  contained  slack")
print("-" * 90)
for parent, child in G.edges:
    px, py = pos_bu[parent]
    cx, cy = pos_bu[child]
    dist = np.sqrt((px - cx) ** 2 + (py - cy) ** 2)
    r_parent = node_radius_bu(parent)
    r_child  = node_radius_bu(child)
    needed = dist + r_child
    slack = r_parent - needed
    ok = slack >= -1e-6
    mark = "✓" if ok else "✗"
    print(f"{parent:25s} -> {child:22s}  {needed:8.3f}      {r_parent:6.3f}  {mark}   {slack:+.3f}")

## Gradient descent — greedy bottom-up init

Same problem setup as the treemap GD section.
The greedy init already satisfies all containment edges (including the DAG edge),
so GD mainly needs to compact the layout and shrink radii.

In [ ]:
fallback_r = float(fixed_node_radii.max()) if len(fixed_node_radii) else 1.0
initial_node_xys_bu = np.stack(
    [pos_bu.get(n, (total_scale / 2, total_scale / 2)) for n in all_node_names],
    dtype=np.float32,
)
initial_variable_radii_bu = np.array(
    [var_radii_bu.get(n, fallback_r) for n in non_leaf_nodes], dtype=np.float32
)

def initialize_bu(_, _seed):
    return {
        "node_xys": initial_node_xys_bu.copy(),
        "variable_node_radii": initial_variable_radii_bu.copy(),
    }

problem_bu = OptimizationProblemTemplate(terms=terms, initialize=initialize_bu).instantiate(
    input_parameters
)
optim_vars_bu, history_bu = problem_bu.optimize(OptimConfig(n_iters=4000, learning_rate=0.005))

pos_gd_bu = {
    node: tuple(float(c) for c in xy)
    for node, xy in zip(all_node_names, optim_vars_bu["node_xys"])
}
var_radii_gd_bu = dict(zip(non_leaf_nodes, [float(r) for r in optim_vars_bu["variable_node_radii"]]))

print("Optimized non-leaf radii (greedy bottom-up init):")
for name, r in sorted(var_radii_gd_bu.items(), key=lambda kv: -kv[1]):
    init_r = var_radii_bu.get(name, float("nan"))
    print(f"  {name:30s}  init={init_r:.3f}  final={r:.3f}  delta={r - init_r:+.3f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 8))

draw_treemap_init(G, pos_bu, var_radii_bu, canvas, axes[0])
axes[0].set_title("Greedy bottom-up initialization\n(before GD)")

draw_circle_layout(G, pos_gd_bu, var_radii_gd_bu, axes[1],
                   title="After gradient descent\n(4000 iters, lr=0.005, greedy init)")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.5))

term_names = ["total_size", "collision", "non_inclusion"]
for ax, name in zip(axes, term_names):
    iters_tm = [h["iteration"] for h in history]
    iters_bu = [h["iteration"] for h in history_bu]
    ax.plot(iters_tm, [h[name] for h in history],    linewidth=1.5, label="treemap init")
    ax.plot(iters_bu, [h[name] for h in history_bu], linewidth=1.5, label="greedy BU init")
    ax.set_title(name)
    ax.set_xlabel("iteration")
    ax.set_ylabel("weighted loss")
    ax.set_yscale("log")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("Loss history: treemap vs greedy bottom-up init", y=1.02)
plt.tight_layout()
plt.show()

# Final containment check
print("\nFinal containment check (greedy BU init):")
print(f"{'Edge':52s}  slack")
print("-" * 65)
for parent, child in G.edges:
    px, py = pos_gd_bu[parent]
    cx, cy = pos_gd_bu[child]
    dist = np.sqrt((px - cx) ** 2 + (py - cy) ** 2)
    r_parent = var_radii_gd_bu.get(parent, G.nodes[parent].get("r", 1.0))
    r_child  = var_radii_gd_bu.get(child,  G.nodes[child].get("r",  1.0))
    slack = r_parent - dist - r_child
    mark = "✓" if slack >= -1e-3 else "✗"
    print(f"{parent:25s} -> {child:22s}  {mark}  {slack:+.4f}")